In [21]:
import pandas as pd
import numpy as np

from matminer.featurizers.structure import SineCoulombMatrix
from pymatgen.core import Element, Composition

print("Libraries imported successfully.")
print("DataFrame shape:", df.shape)

Libraries imported successfully.
DataFrame shape: (545, 4)


In [23]:
# S23 - Sine Coulomb Matrix for 3 structures

from matminer.featurizers.structure import SineCoulombMatrix

# Take 3 structures
df_sample = df[df["structure"].notna()].head(3).copy()

print("Structures selected:", len(df_sample))

# Create Sine Coulomb Matrix
scm = SineCoulombMatrix(flatten=True)

# Calculate features one by one
scm_results = []

for index, structure in df_sample["structure"].items():
    try:
        features = scm.featurize(structure)
        scm_results.append(features)
        print("Structure", index, "processed successfully.")
    except Exception as error:
        print("Structure", index, "skipped:", str(error))
        scm_results.append(None)

print("\nSine Coulomb Matrix calculation completed.")

Structures selected: 3
Structure 0 skipped: Please fit the SineCoulombMatrix before featurizing if using flatten=True.
Structure 1 skipped: Please fit the SineCoulombMatrix before featurizing if using flatten=True.
Structure 2 skipped: Please fit the SineCoulombMatrix before featurizing if using flatten=True.

Sine Coulomb Matrix calculation completed.


In [24]:
def compute_custom_feature(formula):
    try:
        formula = str(formula).replace("\u2007", " ").strip()

        composition = Composition(formula)
        electronegativities = []

        for element in composition.elements:
            element_object = Element(str(element))

            if element_object.X is not None:
                electronegativities.append(float(element_object.X))

        if len(electronegativities) < 2:
            return 0.0

        return max(electronegativities) - min(electronegativities)

    except Exception:
        return np.nan


# Select formula column
if "formula" in df.columns:
    formula_column = "formula"
elif "formula_pretty" in df.columns:
    formula_column = "formula_pretty"
else:
    raise KeyError("Formula column not found.")

print("Using formula column:", formula_column)

# Add custom physical feature
df["custom_feature"] = df[formula_column].apply(compute_custom_feature)

print("Custom physical feature successfully added.")

Using formula column: formula_pretty
Custom physical feature successfully added.


In [25]:
print("==========================================")
print("S23: Sine Coulomb Matrix & Custom Features")
print("==========================================")

print("Original DataFrame shape :", df.shape)
print("SCM structures processed :", len(scm_results))
print("Custom feature added     : Yes")

print("\nCustom Feature: Electronegativity Difference")
print("----------------------------------------------")

display(
    df[[formula_column, "custom_feature"]].head(10)
)

print("\nCustom feature statistics:")
display(df["custom_feature"].describe())

print("\nS23 session completed successfully.")

S23: Sine Coulomb Matrix & Custom Features
Original DataFrame shape : (545, 5)
SCM structures processed : 3
Custom feature added     : Yes

Custom Feature: Electronegativity Difference
----------------------------------------------


,formula_pretty,custom_feature
0,Lu3Sc2Al3O12,2.17
1,Lu3Sc3Al2O12,2.17
2,Y3Sc3Al2O12,2.22
3,MgZrO3,2.13
4,Lu3Al3(GaO6)2,2.17
5,Lu3Sc3(GaO6)2,2.17
6,Y3Al3(GaO6)2,2.22
7,Sr2LuVO6,2.49
8,Sr2ScVO6,2.49
9,Y2MgZrO6,2.22



Custom feature statistics:


count    544.000000
mean       1.193346
std        0.791266
min        0.000000
25%        0.590000
50%        0.850000
75%        2.170000
max        3.190000
Name: custom_feature, dtype: float64


S23 session completed successfully.
